# 10 - Foundry IQ: Web IQ + Content Understanding Blob chunks

## Required permissions

| Identity | Permission | Function |
|---|---|---|
| Signed-in user | **Search Service Contributor** on Search | Author the two sources and independent knowledge base |
| Signed-in user | **Search Index Data Reader** (or **Search Index Data Contributor**) on Search | Inspect and retrieve existing Blob chunks |
| Search managed identity | **Cognitive Services OpenAI User** on the Foundry account | Keyless chat planning/synthesis and existing index vectorizer inference |
| Web IQ profile manager | Authority to issue an authorized profile key | Supply `WEBIQ_API_KEY` for Search's stored-header Web IQ calls |

**Resource prefix:** set `WORKSHOP_PREFIX` in your `.env` (3-20 lowercase letters/digits, starting with a letter).
Prefixes prevent naming collisions, not enforce access isolation. This lab creates only
`<prefix>-nb10-webiq`, `<prefix>-nb10-blob` and `<prefix>-nb10-kb`.

Progress from [09 - Web-only](09-foundry-iq-webiq-only.ipynb) to this **Web + Blob** lab,
then [11 - complete knowledge base](11-foundry-iq-knowledge-base.ipynb).
This notebook is standalone: it does not reuse or change either neighboring lab's definitions.
It requires no SharePoint site, app credential, ACL index, APIM, broker or agent.

Search agentic retrieval, chat inference, embedding/vectorizer inference and Web IQ calls may incur charges.
Stored Web IQ authentication authorizes both portal and notebook retrieval; protect source-author permissions
because the source definition contains the profile key. Do not print the source PUT response or export that key.


Set the kernel working directory to `python-notebooks`. Configuration loads `..\.env` first, then `..\.env.azure`, both with `override=True`; set `WORKSHOP_PREFIX` before running (no prefix prompt).


## Existing resources and configuration

Use an existing Search service and Foundry chat deployment, with the service-identity model grants above.
Complete [05 - Content Understanding](05-blob-content-understanding.ipynb) using the same prefix.
This lab uses its `<prefix>-cu-chunks` index of Content Understanding-enriched Blob documents.
This notebook references that index read-only: it does not upload files or create indexing resources.
The index's existing semantic configuration and managed-identity vectorizer remain unchanged.

Sign in with `az login` in `AZURE_TENANT_ID`. Set the notebook kernel's working directory to
`python-notebooks`. Configuration loads `..\.env` first and `..\.env.azure` second, both overriding
stale process settings. Required settings: `WORKSHOP_PREFIX`, `AZURE_TENANT_ID`, `AZURE_SEARCH_ENDPOINT`,
`AZURE_AI_ACCOUNT_NAME`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`, `WEBIQ_API_KEY`.
`AZURE_AI_MODEL_NAME` defaults to `gpt-5-mini`; it must describe the existing chat deployment.
Configuration selects `f'{prefix}-cu-chunks'` automatically; no index-name change is needed.


In [ ]:
import json
import os
import re
from uuid import UUID

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

load_dotenv(r'..\.env', override=True)
load_dotenv(r'..\.env.azure', override=True)
required = ('WORKSHOP_PREFIX', 'AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'AZURE_AI_ACCOUNT_NAME',
            'AZURE_AI_MODEL_DEPLOYMENT_NAME', 'WEBIQ_API_KEY')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. See .env.example.')
tenant_id = str(UUID(os.environ['AZURE_TENANT_ID']))
prefix = os.environ['WORKSHOP_PREFIX'].strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')

search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
aoai_endpoint = f"https://{os.environ['AZURE_AI_ACCOUNT_NAME']}.openai.azure.com"
chat_deployment = os.environ['AZURE_AI_MODEL_DEPLOYMENT_NAME']
chat_model = os.getenv('AZURE_AI_MODEL_NAME', 'gpt-5-mini')
webiq_api_key = os.environ['WEBIQ_API_KEY']
api_version = '2026-08-01-preview'
blob_index_name = f'{prefix}-cu-chunks'
blob_source_name = f'{prefix}-nb10-blob'
webiq_source_name = f'{prefix}-nb10-webiq'
knowledge_base_name = f'{prefix}-nb10-kb'

credential = AzureCliCredential(tenant_id=tenant_id, process_timeout=60)
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
print('Search:', search_endpoint)
print('Web IQ source:', webiq_source_name)
print('Knowledge base:', knowledge_base_name)
webiq_tools = ['web']
print('Existing Blob index:', blob_index_name)


## 1. Verify the existing Blob index

The read-only GET checks retrievable chunk/source fields and `chunk-semantic`; a count-only
query then verifies nonzero chunks visible to your Search identity. Missing schema or empty/inaccessible
data is a prerequisite failure, not an invitation to rebuild the index here.


In [ ]:
for index_name, lab, semantic_name, content_field, data_fields in [
    (blob_index_name, '05', 'chunk-semantic', 'chunk',
     ['chunk_id', 'chunk', 'title', 'source/file_name', 'source/url']),
]:
    token = search_token()
    headers = {'Authorization': f'Bearer {token}'}
    response = requests.get(
        f'{search_endpoint}/indexes/{index_name}?api-version={api_version}',
        headers=headers, timeout=(10, 90),
    )
    if response.status_code == 404:
        raise RuntimeError(f'Index {index_name} not found. Complete notebook {lab} with the same prefix first.')
    response.raise_for_status()
    index = response.json()
    fields = {field['name']: field for field in index['fields']}
    for field in index['fields']:
        for child in field.get('fields', []):
            fields[f"{field['name']}/{child['name']}"] = child
    if (any(name not in fields or fields[name].get('retrievable') is False for name in data_fields)
            or not fields.get(content_field, {}).get('searchable')):
        raise RuntimeError(f'{index_name} does not have the expected retrievable content/source fields. See notebook {lab}.')
    configurations = index.get('semantic', {}).get('configurations', [])
    if semantic_name not in {config['name'] for config in configurations}:
        raise RuntimeError(f'{index_name} is missing semantic configuration {semantic_name}. See notebook {lab}.')
    response = requests.post(
        f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
        headers=headers, json={'search': '*', 'count': True, 'top': 0}, timeout=(10, 90),
    )
    response.raise_for_status()
    count = response.json()['@odata.count']
    if not isinstance(count, int) or isinstance(count, bool) or count <= 0:
        raise RuntimeError(f'No chunks are visible in {index_name}. Check ingestion and document access in notebook {lab}.')
    print(f'{index_name}: {count} visible chunks; existing schema unchanged.')


## 2. Create exactly two knowledge sources

The Blob `searchIndex` source points to the existing chunk index and its source metadata.
The Web IQ `mcpServer` source calls `https://api.microsoft.ai/v3/mcp` with a stored `x-apikey`
header and only the basic `web` tool allowlisted. No dynamic caller token is sent to Web IQ.
Every Web IQ call must request `contentFormat: passage`; do not send internal document text to Web IQ.
These PUTs create source definitions only, not indexes, indexers, skillsets or data sources.


In [ ]:
blob_source = {
    'name': blob_source_name, 'kind': 'searchIndex',
    'description': 'Travel reimbursement and onboarding policy documents from the existing Blob index.',
    'searchIndexParameters': {
        'searchIndexName': blob_index_name,
        'semanticConfigurationName': 'chunk-semantic',
        'searchFields': [{'name': 'chunk'}],
        'sourceDataFields': [
            {'name': name} for name in ('chunk_id', 'chunk', 'title', 'source/file_name', 'source/url')
        ],
    },
}
response = requests.put(
    f'{search_endpoint}/knowledgesources/{blob_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=blob_source, timeout=(10, 90),
)
response.raise_for_status()
print('Blob knowledge source ready:', blob_source_name, '->', blob_index_name)


In [ ]:
webiq_source = {
    'name': webiq_source_name,
    'kind': 'mcpServer',
    'description': (
        'Basic Web IQ web search for public information. For every web call, explicitly set '
        'contentFormat to passage; use query-dependent passages, not text, html or markdown.'
    ),
    'resultsProcessing': 'rerank',
    'mcpServerParameters': {
        'serverURL': 'https://api.microsoft.ai/v3/mcp',
        'authentication': {
            'kind': 'storedHeaders',
            'storedHeadersParameters': {'headers': {'x-apikey': webiq_api_key}},
        },
        'tools': [{'name': 'web', 'outputParsing': {'kind': 'auto'}, 'maxOutputTokens': 4000}],
    },
}
response = requests.put(
    f'{search_endpoint}/knowledgesources/{webiq_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=webiq_source, timeout=(10, 120),
)
response.raise_for_status()
print('Web IQ source ready:', webiq_source_name, '| authentication: storedHeaders | tool: web')


## 3. Create the independent Web + Blob knowledge base

The existing Foundry chat deployment handles query planning and answer synthesis via Search's
managed identity; no model API key is embedded. `low` reasoning effort keeps the initial comparison
small. The KB contains exactly the two sources above. Its routing instructions distinguish internal
policy evidence from public web evidence without disclosing internal passages to Web IQ.


In [ ]:
knowledge_base = {
    'name': knowledge_base_name,
    'description': 'Existing Blob policies plus live public web evidence from Web IQ.',
    'retrievalInstructions': (
        f'Use {blob_source_name} for internal travel reimbursement and onboarding policies. '
        f'Use {webiq_source_name} for current public information and external facts. '
        'For comparisons use both sources. Never send internal document text to Web IQ. '
        'For every web call explicitly set contentFormat to passage. '
        'Use specific public search terms, locations and dates from the question.'
    ),
    'answerInstructions': (
        'Answer concisely and cite supporting reference IDs. Distinguish internal policy facts '
        'from public web facts. If the evidence is missing, say you do not know.'
    ),
    'outputMode': 'answerSynthesis',
    'knowledgeSources': [{'name': blob_source_name}, {'name': webiq_source_name}],
    'models': [{
        'kind': 'azureOpenAI',
        'azureOpenAIParameters': {
            'resourceUri': aoai_endpoint,
            'deploymentId': chat_deployment,
            'modelName': chat_model,
        },
    }],
    'retrievalReasoningEffort': {'kind': 'low'},
}
response = requests.put(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=knowledge_base, timeout=(10, 90),
)
response.raise_for_status()
print('Web IQ + Blob knowledge base ready:', knowledge_base_name)


## 4. Retrieve Blob evidence, web evidence, then compare both

Each cell shows the actual retrieve request and its source-specific `knowledgeSourceParams`.
Start with `answerSynthesis` to see a grounded answer, then rerun the same question with
`output_mode = 'extractiveData'` to inspect retrieved evidence without final-answer synthesis.
Keep `low` KB reasoning effort for the first comparison; optionally change it to `medium` in the
KB PUT and compare activity, latency and evidence coverage. Extractive output can still require
chat inference for planning; it is not a guarantee of no model usage.

Inspect activity for the source actually queried, and references for the supporting passage and URL.
Mixed retrieval makes both sources available; the planner may choose only one if evidence is missing.
Do not infer that both were used from an answer alone, or treat an empty reference list as proof of
an authentication failure. HTTP 206 or any source activity error is rejected, even when an answer exists.


In [ ]:
output_mode = 'answerSynthesis'  # Compare with 'extractiveData' on the same questions.


In [ ]:
question = f'Using {blob_source_name}, what receipts do I need for travel reimbursement?'
token = search_token()
response = requests.post(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={'Authorization': f'Bearer {token}'},
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'outputMode': output_mode,
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [{
            'knowledgeSourceName': blob_source_name, 'kind': 'searchIndex',
            'includeReferences': True, 'includeReferenceSourceData': True,
        }],
    }, timeout=(10, 120),
)
print('HTTP status:', response.status_code)
print('Request ID:', response.headers.get('x-ms-request-id') or response.headers.get('request-id'))
if response.status_code >= 400:
    print(response.text.replace(webiq_api_key, '<redacted>').replace(token, '<redacted>')[:8000])
response.raise_for_status()
result = response.json()

print('\nActivity:')
print(json.dumps(result.get('activity', []), indent=2).replace(webiq_api_key, '<redacted>').replace(token, '<redacted>'))
print('\nAnswer or extractive output:')
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text', '').replace(webiq_api_key, '<redacted>').replace(token, '<redacted>'))
print('\nReferences and retrieved passages:')
for reference in result.get('references', []):
    source_data = reference.get('sourceData') or {}
    try:
        parsed = json.loads(source_data.get('content') or '{}')
        if isinstance(parsed, dict):
            source_data = {**source_data, **parsed}
    except (TypeError, ValueError):
        pass
    print('-', reference.get('id'), str(source_data.get('title', '')).replace(webiq_api_key, '<redacted>').replace(token, '<redacted>'),
          str(source_data.get('url') or source_data.get('source/url') or (source_data.get('source') or {}).get('url') or '').replace(webiq_api_key, '<redacted>').replace(token, '<redacted>'))
    print(str(source_data.get('chunk') or source_data.get('content') or '').replace(webiq_api_key, '<redacted>').replace(token, '<redacted>')[:2000])
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity above.')
if not result.get('references'):
    print('No references returned. Inspect activity; this alone does not establish an authentication failure.')


In [ ]:
question = 'What are the current public IRS guidelines for business travel receipt records?'
token = search_token()
response = requests.post(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={'Authorization': f'Bearer {token}'},
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'outputMode': output_mode,
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [{
            'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer',
            'includeReferences': True, 'includeReferenceSourceData': True,
        }],
    }, timeout=(10, 120),
)
print('HTTP status:', response.status_code)
print('Request ID:', response.headers.get('x-ms-request-id') or response.headers.get('request-id'))
if response.status_code >= 400:
    print(response.text.replace(webiq_api_key, '<redacted>').replace(token, '<redacted>')[:8000])
response.raise_for_status()
result = response.json()

print('\nActivity:')
print(json.dumps(result.get('activity', []), indent=2).replace(webiq_api_key, '<redacted>').replace(token, '<redacted>'))
print('\nAnswer or extractive output:')
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text', '').replace(webiq_api_key, '<redacted>').replace(token, '<redacted>'))
print('\nReferences and retrieved passages:')
for reference in result.get('references', []):
    source_data = reference.get('sourceData') or {}
    try:
        parsed = json.loads(source_data.get('content') or '{}')
        if isinstance(parsed, dict):
            source_data = {**source_data, **parsed}
    except (TypeError, ValueError):
        pass
    print('-', reference.get('id'), str(source_data.get('title', '')).replace(webiq_api_key, '<redacted>').replace(token, '<redacted>'),
          str(source_data.get('url') or source_data.get('source/url') or (source_data.get('source') or {}).get('url') or '').replace(webiq_api_key, '<redacted>').replace(token, '<redacted>'))
    print(str(source_data.get('chunk') or source_data.get('content') or '').replace(webiq_api_key, '<redacted>').replace(token, '<redacted>')[:2000])
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity above.')
if not result.get('references'):
    print('No references returned. Inspect activity; this alone does not establish an authentication failure.')


In [ ]:
question = f'Use {blob_source_name} for our internal travel receipt policy and {webiq_source_name} for public IRS travel receipt guidelines. Compare the requirements; do not send internal text to the web.'
token = search_token()
response = requests.post(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={'Authorization': f'Bearer {token}'},
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'outputMode': output_mode,
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': 'searchIndex',
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer',
             'includeReferences': True, 'includeReferenceSourceData': True},
        ],
    }, timeout=(10, 120),
)
print('HTTP status:', response.status_code)
print('Request ID:', response.headers.get('x-ms-request-id') or response.headers.get('request-id'))
if response.status_code >= 400:
    print(response.text.replace(webiq_api_key, '<redacted>').replace(token, '<redacted>')[:8000])
response.raise_for_status()
result = response.json()

print('\nActivity:')
print(json.dumps(result.get('activity', []), indent=2).replace(webiq_api_key, '<redacted>').replace(token, '<redacted>'))
print('\nAnswer or extractive output:')
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text', '').replace(webiq_api_key, '<redacted>').replace(token, '<redacted>'))
print('\nReferences and retrieved passages:')
for reference in result.get('references', []):
    source_data = reference.get('sourceData') or {}
    try:
        parsed = json.loads(source_data.get('content') or '{}')
        if isinstance(parsed, dict):
            source_data = {**source_data, **parsed}
    except (TypeError, ValueError):
        pass
    print('-', reference.get('id'), str(source_data.get('title', '')).replace(webiq_api_key, '<redacted>').replace(token, '<redacted>'),
          str(source_data.get('url') or source_data.get('source/url') or (source_data.get('source') or {}).get('url') or '').replace(webiq_api_key, '<redacted>').replace(token, '<redacted>'))
    print(str(source_data.get('chunk') or source_data.get('content') or '').replace(webiq_api_key, '<redacted>').replace(token, '<redacted>')[:2000])
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity above.')
if not result.get('references'):
    print('No references returned. Inspect activity; this alone does not establish an authentication failure.')


## Diagnose evidence and authorization boundaries

HTTP failures include a request ID and redacted error body. Activity errors also fail the cell;
check `knowledgeSourceName`, counts and errors before calling the answer successful.
Web IQ `auth_missing_token` points to the stored profile header/authorization, not your Search token.
Search 403 on index reads indicates your Search data role; chat/vectorizer authorization failures
require checking Search's managed-identity model grant. HTTP 429 indicates quota or throttling.
An empty Content Understanding Blob index requires completing 05. Sparse web references may require a more precise
public query; inspect reference URLs and source passages rather than assuming an authentication error.

The portal can retrieve through this same KB using the stored Web IQ header. Protect source-author
access and never display the echoed Web IQ source definition, stored key or bearer token.
After comparing both output modes, continue to [11 - Web + Blob + SharePoint](11-foundry-iq-knowledge-base.ipynb)
when its separate SharePoint indexing and user-authorization prerequisites are ready.


### Web IQ + Content Understanding Blob knowledge base in the portal

![Foundry IQ portal showing active Blob and Web IQ sources, a cited public Python-release answer and an internal reimbursement-policy answer](../docs/webiq-blob-knowledge-base-portal.png)

The left pane shows the two active knowledge sources: `<prefix>-nb10-blob`, which references lab 05's `<prefix>-cu-chunks` index, and `<prefix>-nb10-webiq`. The public Python-release question uses Web IQ references; the reimbursement-policy question demonstrates retrieval from the internal travel handbook. Inspect each answer's references and Debug activity to confirm which source supplied its evidence.

The screenshot is illustrative, not a current Python-version reference or a required answer. Your resource prefix, release dates, retrieved passages and responses can differ. These are two separate questions, not proof that one request combined both sources; use the notebook's mixed-question cell to test that. Sending questions in the portal makes additional live retrieval requests and can incur model and Web IQ charges.

## Optional cleanup

Keep `CLEAN_UP = False` to compare results later. Opting in deletes this KB first and then only
its two source definitions. It does not delete the existing Blob index, files or neighboring labs.


In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    for collection, name in (('knowledgebases', knowledge_base_name),
                             ('knowledgesources', webiq_source_name),
                             ('knowledgesources', blob_source_name)):
        response = requests.delete(
            f'{search_endpoint}/{collection}/{name}?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        if response.status_code != 404:
            response.raise_for_status()
        print('Removed or already absent:', name)
else:
    print('Cleanup disabled; Web IQ + Blob resources remain available.')


## References

- [MCP server knowledge sources and authentication](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-mcp-server)
- [Create a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Retrieve action and response](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve)
- [Answer synthesis and per-request output mode](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-answer-synthesis)
